# Day 5: Quantitative Structural Connectomics & Graph Theory

Welcome to the **Day 5 Interactive Connectomics Laboratory**.

In this notebook, we analyze whole-brain structural connectomes generated during **Tutorial 5.4**.

We explore:
1. **Raw vs. SIFT2-Weighted Connectivity Matrices**: The impact of spherical deconvolution streamline filtering.
2. **Network Topology & Graph Theory with NetworkX**: Graph density, global efficiency, weighted clustering coefficients, and identification of network hubs.

In [ ]:
import os
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
%matplotlib inline

def find_file(filename):
    candidates = [filename, os.path.join("..", filename)]
    for path in candidates:
        if os.path.exists(path):
            return path
    return None

raw_csv = find_file("connectome.csv")
sift2_csv = find_file("connectome_sift2.csv")

print(f"Raw Connectome Matrix   : {raw_csv}")
print(f"SIFT2 Connectome Matrix : {sift2_csv}")

In [ ]:
if sift2_csv and os.path.exists(sift2_csv):
    sift2_mat = np.loadtxt(sift2_csv, delimiter=",")
    raw_mat = np.loadtxt(raw_csv, delimiter=",") if raw_csv and os.path.exists(raw_csv) else None
    
    fig, axes = plt.subplots(1, 2 if raw_mat is not None else 1, figsize=(14 if raw_mat is not None else 7, 6))
    if raw_mat is None:
        axes = [axes]
    
    if raw_mat is not None:
        im0 = axes[0].imshow(np.log1p(raw_mat), cmap="viridis")
        axes[0].set_title("Raw Streamline Counts (log1p)", fontsize=12, fontweight="bold")
        axes[0].set_xlabel("Node Index")
        axes[0].set_ylabel("Node Index")
        plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04, label="log(1 + Streamlines)")
    
    idx = 1 if raw_mat is not None else 0
    im1 = axes[idx].imshow(np.log1p(sift2_mat), cmap="magma")
    axes[idx].set_title("SIFT2 Cross-Section Weighted (log1p)", fontsize=12, fontweight="bold")
    axes[idx].set_xlabel("Node Index")
    axes[idx].set_ylabel("Node Index")
    plt.colorbar(im1, ax=axes[idx], fraction=0.046, pad=0.04, label="log(1 + SIFT2 Weight)")
    
    plt.tight_layout()
    plt.show()
    
    # Graph Theory Analysis with NetworkX
    G = nx.from_numpy_array(sift2_mat)
    print("\n--- Network Topology Metrics (SIFT2 Connectome) ---")
    print(f"  Total Nodes             : {G.number_of_nodes()}")
    print(f"  Total Edges             : {G.number_of_edges()}")
    print(f"  Graph Density           : {nx.density(G):.4f}")
    print(f"  Global Efficiency       : {nx.global_efficiency(G):.4f}")
    print(f"  Weighted Avg Clustering : {nx.average_clustering(G, weight='weight'):.4f}")
    
    # Hub Identification
    strengths = dict(G.degree(weight="weight"))
    top_hubs = sorted(strengths.items(), key=lambda x: x[1], reverse=True)[:5]
    print("\n--- Top 5 Hub Nodes by Connection Strength ---")
    for node, strength in top_hubs:
        print(f"  Node {node:02d} : Total Strength = {strength:.2f}")
else:
    print("Connectome CSV not found yet. Please run tutorial_5.4_capstone_analysis.sh first.")